# 03 · Planning config (single source of truth for parameters)

Creates **`monitoring.planning_config`** — the governed, joinable registry of tunable parameters used by detection (twins + events) **and** the planning engine (inventory policy, MRP, ...).

**Why a table (not a file):** planning parameters get *joined onto items* (service level per material/group/plant, MOQ per material, ...) and support **scope-based overrides**; a table is queryable by notebooks, the app, Genie, and the agent, and Delta gives change history for free.

**Design:** key–value with a `scope_type` / `scope_value`; resolution is **most-specific-scope-wins** (`MATERIAL` > `MATERIAL_GROUP` > `PLANT` > `GLOBAL`). Seeded here with **GLOBAL defaults** (the source of truth, in git); add override rows later as needed. Consumers read via `cfg(key, default)` (fallback keeps notebooks runnable even before this table exists).

**Not here:** data-derived params (lead time, demand mean/variability) stay in the twins — this table holds only *business/policy* choices + detection thresholds.

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
print("planning config target:", f"{FQ_MON}.planning_config")

## Seed GLOBAL defaults
Idempotent overwrite of the GLOBAL rows. `(param_key, value_double | value_string, unit, description)`. Detection thresholds mirror the values previously hard-coded in the twin/event notebooks; the inventory-policy block are business choices the planning engine consumes.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType)

# (param_key, value_double, value_string, unit, description)
DEFAULTS = [
    # --- detection thresholds (consumed by twins + events) ---
    ("low_cover_days", 14.0, None, "days", "Inventory days-of-cover below this = low cover (is_low_cover)"),
    ("overstock_days", 90.0, None, "days", "Inventory days-of-cover above this = overstock (is_overstock)"),
    ("procurement_risk_otif", 0.80, None, "fraction", "Supplier OTIF below this flags supplier_risk_flag"),
    ("procurement_risk_rejection", 0.05, None, "fraction", "Supplier rejection rate above this flags supplier_risk_flag"),
    ("distribution_risk_otif", 0.90, None, "fraction", "Distribution OTIF below this flags fulfillment_risk_flag"),
    ("low_yield_pct", 90.0, None, "percent", "Production yield %% below this raises LOW_YIELD"),
    ("low_fill_rate", 0.95, None, "fraction", "Distribution fill rate below this raises LOW_FILL"),
    ("forecast_drift_wmape", 0.60, None, "fraction", "Forecast WMAPE above this raises FORECAST_DRIFT"),
    # --- inventory-policy defaults (consumed by the planning engine) ---
    ("service_level", 0.95, None, "fraction", "Target cycle service level (drives safety-stock z)"),
    ("review_period_days", 7.0, None, "days", "Review period R for periodic-review policy"),
    ("min_order_qty", 0.0, None, "units", "Minimum order quantity (MOQ)"),
    ("order_multiple", 1.0, None, "units", "Order rounding multiple"),
    ("holding_cost_rate", 0.25, None, "fraction_per_year", "Annual holding cost rate (for EOQ)"),
    ("ordering_cost", 50.0, None, "currency", "Fixed cost per order (for EOQ)"),
    ("safety_stock_method", None, "normal", "method", "Safety-stock model: normal | none"),
    ("lead_time_source", None, "procurement_twin", "source", "Where lead time is read from (data-derived twin)"),
]

schema = StructType([
    StructField("param_key", StringType()), StructField("value_double", DoubleType()),
    StructField("value_string", StringType()), StructField("unit", StringType()),
    StructField("description", StringType()),
])
rows = spark.createDataFrame([(k, vd, vs, u, d) for (k, vd, vs, u, d) in DEFAULTS], schema)
seed = (rows
    .withColumn("scope_type", F.lit("GLOBAL"))
    .withColumn("scope_value", F.lit(None).cast("string"))
    .withColumn("updated_by", F.lit("seed"))
    .withColumn("updated_at", F.current_timestamp())
    .select("param_key", "scope_type", "scope_value", "value_double", "value_string",
            "unit", "description", "updated_by", "updated_at"))

(seed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ_MON}.planning_config"))
spark.sql(f"ALTER TABLE {FQ_MON}.planning_config SET TBLPROPERTIES ('comment' = 'Single source of truth for tunable planning + detection parameters. Key-value with scope-based overrides (MATERIAL > MATERIAL_GROUP > PLANT > GLOBAL).')")
print("seeded", seed.count(), "GLOBAL params")

## Column descriptions

In [ ]:
COMMENTS = {
  "param_key": "Parameter name (e.g. service_level, low_cover_days)",
  "scope_type": "Scope of this value: GLOBAL | MATERIAL_GROUP | PLANT | MATERIAL | SUPPLIER",
  "scope_value": "The scope key (NULL for GLOBAL; else group/plant/material/supplier id)",
  "value_double": "Numeric value (NULL for string params)",
  "value_string": "String value (NULL for numeric params)",
  "unit": "Unit of the value (days, fraction, percent, units, currency, ...)",
  "description": "What the parameter controls",
  "updated_by": "Who last set this value (seed | user | agent)",
  "updated_at": "When this value was last set",
}
for col, c in COMMENTS.items():
    spark.sql(f"ALTER TABLE {FQ_MON}.planning_config ALTER COLUMN {col} COMMENT '" + c.replace("'", "''") + "'")
print("column comments set:", len(COMMENTS))

## Resolver view `v_resolved_config`
Effective **GLOBAL** value per `param_key` (latest write wins). For per-entity resolution with overrides, join your item list to `planning_config` and rank by scope precedence (pattern below).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {FQ_MON}.v_resolved_config (
  param_key    COMMENT 'Parameter name',
  value_double COMMENT 'Effective GLOBAL numeric value',
  value_string COMMENT 'Effective GLOBAL string value',
  unit         COMMENT 'Unit',
  description  COMMENT 'What the parameter controls'
) COMMENT 'Effective GLOBAL planning/detection parameters (most recent write per key).' AS
SELECT param_key, value_double, value_string, unit, description FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY param_key ORDER BY updated_at DESC) rn
  FROM {FQ_MON}.planning_config WHERE scope_type = 'GLOBAL'
) WHERE rn = 1
""")
print("created", f"{FQ_MON}.v_resolved_config")

# Per-entity override resolution pattern (for later, when non-GLOBAL rows exist):
#   SELECT items.material, c.value_double
#   FROM items LEFT JOIN planning_config c
#     ON c.param_key = 'service_level'
#    AND ( (c.scope_type='MATERIAL'       AND c.scope_value = items.material)
#       OR (c.scope_type='MATERIAL_GROUP' AND c.scope_value = items.material_group)
#       OR (c.scope_type='PLANT'          AND c.scope_value = items.plant)
#       OR (c.scope_type='GLOBAL') )
#   QUALIFY ROW_NUMBER() OVER (PARTITION BY items.material ORDER BY
#     CASE c.scope_type WHEN 'MATERIAL' THEN 0 WHEN 'MATERIAL_GROUP' THEN 1 WHEN 'PLANT' THEN 2 ELSE 3 END) = 1'''

## `cfg()` helper (copy into consumers)
Reads an effective GLOBAL value with a safe fallback — so twins/events/planning stay runnable even if this table hasn't been built yet.

In [ ]:
def cfg(key, default, kind="double"):
    """Effective GLOBAL param with fallback to `default` if the table/param is missing."""
    col = "value_string" if kind == "string" else "value_double"
    try:
        r = spark.sql(f"SELECT {col} FROM {FQ_MON}.v_resolved_config WHERE param_key = '{key}'").collect()
        return r[0][0] if r and r[0][0] is not None else default
    except Exception:
        return default

# quick check
print("service_level     =", cfg("service_level", 0.95))
print("low_cover_days    =", cfg("low_cover_days", 14))
print("safety_stock_method =", cfg("safety_stock_method", "normal", kind="string"))

## Verify

In [ ]:
display(spark.sql(f"SELECT param_key, value_double, value_string, unit, description FROM {FQ_MON}.v_resolved_config ORDER BY param_key"))

## Next
With the registry in place, every downstream consumer reads one source via `cfg(...)`. **Next: build the digital twins** (`04_demand_twin` → `08_distribution_twin`), whose stockout / low-cover / OTIF / yield thresholds resolve through this table; then the **events** notebook (`13_events`) and the **planning engines** (inventory policy `14`, MRP `15`) read the same `service_level` / `review_period_days` / MOQ parameters.